In [0]:

-- 1. Staging view to cast types and normalize dates
CREATE STREAMING LIVE VIEW stg_returns AS
SELECT
  -- Handle date formatting anomalies
  COALESCE(
    try_to_date(return_date, 'MM/dd/yyyy'),
    try_to_date(return_date, 'yyyy-MM-dd'),
    try_to_date(return_date, 'dd-MMM-yy')
  ) AS return_date,
  
  -- Casting bad string data ("UNKNOWN", "five") automatically converts them to NULL
  CAST(product_id AS INT) AS product_id,
  CAST(store_id AS INT) AS store_id,
  CAST(quantity AS INT) AS quantity,
  
  ingested_timestamp,
  batch_date
FROM STREAM(mavenmarket.bronze.returns);

-- 2. Clean Silver Table filtering out anomalies
CREATE STREAMING LIVE TABLE silver_returns (
  CONSTRAINT valid_quantity EXPECT (quantity > 0) ON VIOLATION DROP ROW,
  CONSTRAINT valid_store EXPECT (store_id IS NOT NULL) ON VIOLATION DROP ROW,
  CONSTRAINT valid_product EXPECT (product_id IS NOT NULL) ON VIOLATION DROP ROW
)
COMMENT "Cleaned returns fact table"
AS SELECT * FROM STREAM(live.stg_returns);